# Assessment: Clasificación de Biomarcadores para Alzheimer

## Contexto

El Alzheimer (AD) es una enfermedad neurodegenerativa que actualmente se diagnostica principalmente por síntomas clínicos, cuando el daño neuronal ya es significativo. El framework **AT(N)** — Amyloid (A), Tau (T), Neurodegeneration (N) — permite categorizar biomarcadores según el proceso patológico que reflejan, abriendo la puerta a diagnósticos más tempranos basados en biología.

En este assessment trabajarás con un dataset sintético de biomarcadores de **líquido cefalorraquídeo (CSF)** y **sangre (plasma)** para clasificar pacientes en tres categorías:

- **NC** — Normal Cognition (cognición normal)
- **MCI** — Mild Cognitive Impairment (deterioro cognitivo leve)
- **AD** — Alzheimer's Disease

### Lo que evaluamos

Buscamos entender **cómo piensas**, **cómo abordas problemas**, y **cómo comunicas tus decisiones**.

---

## Instrucciones

1. Completa las **tres partes** de este notebook.
2. Escribe tu código en las celdas indicadas.
3. Responde las preguntas abiertas en celdas de Markdown.
4. Cuando termines, haz **push** a tu fork y envía la liga del repositorio.

**Puedes usar:** documentación, Google, recursos en línea. Evita usar IA para generar el código. Preferimos ver TU razonamiento :)

---
## Setup

Ejecuta esta celda para cargar las dependencias y el dataset. **No modifiques esta sección.**

In [ ]:
# ============================================================
# SETUP — No modificar esta celda
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    ConfusionMatrixDisplay, accuracy_score, f1_score
)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Carga del dataset
df = pd.read_csv('../data/biomarker_data.csv')
print(f"Dataset cargado: {df.shape[0]} muestras, {df.shape[1]} columnas")
print(f"\nPrimeras filas:")
df.head()

### Variables del dataset

| Variable | Fuente | Descripción |
|---|---|---|
| `CSF_AB42` | CSF | Beta-amiloide 1-42 (pg/mL) — marcador de amiloidosis |
| `CSF_TAU` | CSF | Tau total (pg/mL) — marcador de neurodegeneración |
| `CSF_PTAU` | CSF | Tau fosforilada (pg/mL) — marcador de taupatía |
| `PLASMA_PTAU217` | Sangre | Fosfo-tau 217 (pg/mL) — marcador emergente de tau |
| `PLASMA_AB42` | Sangre | Beta-amiloide 42 en plasma (pg/mL) |
| `PLASMA_AB40` | Sangre | Beta-amiloide 40 en plasma (pg/mL) |
| `PLASMA_NFL` | Sangre | Neurofilamento ligero (pg/mL) — daño axonal |
| `PLASMA_GFAP` | Sangre | Proteína ácida fibrilar glial (pg/mL) — astrogliosis |
| `AGE` | Demográfica | Edad del paciente |
| `SEX` | Demográfica | Sexo (0=Femenino, 1=Masculino) |
| `EDUCATION_YEARS` | Demográfica | Años de educación |
| `PLASMA_AB42_AB40_RATIO` | Derivada | Ratio Aβ42/Aβ40 en plasma |
| `PLASMA_PTAU217_AB42_RATIO` | Derivada | Ratio pTau217/Aβ42 en plasma |
| `DIAGNOSIS` | Target | NC, MCI, o AD |

---
# PARTE 1: Exploración de Datos (30%)

Antes de construir cualquier modelo, necesitas entender los datos con los que trabajas.

### 1.1 Análisis descriptivo

Explora el dataset y responde:
- ¿Cuántas muestras hay por clase?
- ¿Hay valores faltantes? ¿En qué columnas?
- ¿Cuáles son las estadísticas descriptivas de los biomarcadores por grupo diagnóstico?

In [ ]:
# ============================================================
# Pon tu código del análisis descriptivo aquí 
# ============================================================
# Sugerencias:
# - df['DIAGNOSIS'].value_counts()
# - df.isnull().sum()
# - df.groupby('DIAGNOSIS').describe()



### 1.2 Visualización exploratoria

Crea al menos **dos visualizaciones** que te ayuden a entender la relación entre los biomarcadores y el diagnóstico. Por ejemplo: distribuciones por clase, boxplots, scatter plots, heatmap de correlaciones, etc.

In [ ]:
# ============================================================
# Pon tu código de las Visualizaciones aquí
# ============================================================



### 1.3 Pregunta abierta

Basándote en tu exploración, responde en la celda de abajo:

1. **¿Qué biomarcadores parecen ser los más útiles para distinguir entre las tres clases? ¿Por qué?**
2. **¿Qué desafíos anticipas para clasificar correctamente la clase MCI?**
3. **¿Cómo planeas manejar los valores faltantes y por qué elegiste ese enfoque?**

*Tu respuesta aquí:*



---
# PARTE 2: Modelado (40%)

En esta sección vas a entrenar modelos de clasificación. Te proporcionamos un **modelo baseline con overfitting intencional**. Tu tarea es entenderlo, identificar los problemas, y proponer una mejora.

### 2.1 Preparación de datos

Prepara los datos para modelado: maneja los valores faltantes, selecciona features, y separa features de target.

In [ ]:
# ============================================================
# Pon tu códgo de la preparación de datos aquí
# ============================================================
# Maneja valores faltantes, selecciona features, codifica target.
# 
# Pista: necesitas definir X (features) y y (target)
# Recuerda excluir PATIENT_ID y DIAGNOSIS de las features.



### 2.2 Modelo Baseline (overfitting intencional)

Ejecuta el siguiente modelo baseline. Este modelo tiene **problemas intencionales** que deberás identificar.

In [ ]:
# ============================================================
# BASELINE — Ejecuta esta celda sin modificar
# ============================================================
# NOTA: Este modelo tiene problemas intencionales.
# Tu tarea en 2.3 es identificarlos y proponer mejoras.

# Preparación rápida del baseline (usa dropna para simplificar)
df_baseline = df.dropna().copy()
feature_cols = [c for c in df_baseline.columns if c not in ['PATIENT_ID', 'DIAGNOSIS']]
X_base = df_baseline[feature_cols].values
y_base = LabelEncoder().fit_transform(df_baseline['DIAGNOSIS'])

# Random Forest con hiperparámetros que promueven overfitting
rf_overfit = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,        # Sin límite de profundidad
    min_samples_split=2,   # Mínimo para split
    min_samples_leaf=1,    # Mínimo en hojas
    max_features=None,     # Usa TODAS las features en cada split
    class_weight=None,     # No ajusta por desbalance
    random_state=42
)

# Evaluación con cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_base = cross_validate(
    rf_overfit, X_base, y_base, cv=cv,
    scoring=['accuracy', 'f1_macro'],
    return_train_score=True
)

print("="*55)
print("BASELINE MODEL — Random Forest (overfit config)")
print("="*55)
print(f"Train Accuracy:      {scores_base['train_accuracy'].mean():.4f} ± {scores_base['train_accuracy'].std():.4f}")
print(f"Validation Accuracy: {scores_base['test_accuracy'].mean():.4f} ± {scores_base['test_accuracy'].std():.4f}")
print(f"Train F1 (macro):    {scores_base['train_f1_macro'].mean():.4f} ± {scores_base['train_f1_macro'].std():.4f}")
print(f"Validation F1 (macro): {scores_base['test_f1_macro'].mean():.4f} ± {scores_base['test_f1_macro'].std():.4f}")
print(f"\nGap train-val accuracy: {(scores_base['train_accuracy'].mean() - scores_base['test_accuracy'].mean()):.4f}")
print("\n⚠️  Observa el gap entre train y validation. ¿Qué indica esto?")

### 2.3 Tu modelo mejorado

Ahora es tu turno. Entrena un modelo que mejore sobre el baseline. Puedes:

- Ajustar hiperparámetros del Random Forest
- Probar un modelo diferente (XGBoost, Logistic Regression, SVM, etc.)
- Aplicar técnicas de preprocesamiento (normalización, feature selection, etc.)
- Manejar el desbalance de clases de forma diferente

**Requisito:** Usa validación cruzada estratificada (ya importada) y reporta las mismas métricas que el baseline para que sea comparable.

In [ ]:
# ============================================================
# Pon tu código del Modelo mejorado aquí 
# ============================================================
# Entrena tu modelo, evalúalo con cross-validation estratificada,
# y reporta: train accuracy, validation accuracy, F1 score.



### 2.4 Comparación

Presenta una comparación clara entre el baseline y tu modelo. Puede ser una tabla, un gráfico, o ambos.

In [ ]:
# ============================================================
# Ponn tu código de la Comparación baseline vs. tu modelo aquí
# ============================================================



---
# PARTE 3: Análisis Crítico (30%)

Esta sección evalúa tu capacidad de interpretar resultados y pensar más allá del código.

### 3.1 Feature Importance

Usando tu modelo (o el baseline), extrae y visualiza la importancia de cada feature. ¿Coincide con lo que observaste en la exploración?

In [ ]:
# ============================================================
# Pon tu código de las Feature importance aquí
# ============================================================



### 3.2 Análisis de errores

Genera una **matriz de confusión** de tu modelo y analiza:
- ¿Qué clase es más difícil de clasificar?
- ¿Qué tipo de errores son más frecuentes (falsos positivos vs. falsos negativos)?
- ¿Tiene implicaciones clínicas el tipo de error más frecuente?

In [ ]:
# ============================================================
# Pon tu código de la Matriz de confusión aquí
# ============================================================



### 3.3 Preguntas abiertas finales

Responde las siguientes preguntas en la celda de abajo:

1. **¿Qué problemas específicos identificaste en el modelo baseline y qué cambios hiciste para abordarlos?**

2. **Si tuvieras acceso a variables clínicas adicionales (ej: MMSE, CDR, ApoE genotype), ¿cuáles incluirías y por qué?**

3. **En un contexto clínico real, ¿qué tipo de error sería más peligroso: clasificar a un paciente con AD como NC, o clasificar a un paciente NC como AD? ¿Cómo ajustarías tu modelo para reflejar esto?**

4. **¿Qué harías diferente si tuvieras más tiempo? Describe al menos dos mejoras concretas.**

*Tu respuesta aquí:*



---
## Entrega

1. Asegúrate de que todo el notebook corre de principio a fin sin errores.
2. Haz commit y push a tu fork.
3. Envía la liga de tu repositorio a Angel Peña (6271506213).

**¡GenoBit te da las gracias por participar!** 